## COMP90024 Team 2



# Frontend API smoke test

This notebook checks that the frontend can read cleaned data through the single Fission API:

`GET http://localhost:9090/api/query`

Keep this running in another PowerShell before executing the notebook:

```powershell
kubectl port-forward -n fission service/router 9090:80
```

Set `ANALYTICS_API_URL` to use a different endpoint. Analysis requires a populated
cleaned-post index with city/date, sentiment and matched weather fields. Empty
required windows stop with an explanatory message. API connectivity alone does
not establish harvester health or a causal weather effect.


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

import os
API_BASE = os.environ.get("ANALYTICS_API_URL", "http://localhost:9090/api/query")

def api_get(**params):
    response = requests.get(API_BASE, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

plt.style.use("default")
pd.set_option("display.max_colwidth", 120)

def require_frame(frame, columns, context):
    """Stop with actionable input requirements before constructing a plot."""
    missing = sorted(set(columns) - set(frame.columns))
    if frame.empty or missing:
        raise RuntimeError(
            f'{context}: no usable records or missing fields {missing}. '
            'Check API connectivity, the cleaned index, weather joins and date filters.'
        )
    return frame


## 1. Health check

In [ ]:
health = api_get(mode="health")
health

## 2. Summary from posts_clean

In [ ]:
summary = api_get(mode="summary")
if not sum(summary.get('by_city', {}).values()):
    raise RuntimeError('The cleaned-post index is empty. Run ingestion and cleaning before this analysis.')
summary

In [ ]:
city_counts = pd.Series(summary["by_city"], name="posts").sort_values(ascending=False)
platform_counts = pd.Series(summary["by_platform"], name="posts").sort_values(ascending=False)
sentiment_counts = pd.Series(summary["by_sentiment"], name="posts").sort_values(ascending=False)

display(pd.DataFrame({"city_posts": city_counts}))
display(pd.DataFrame({"platform_posts": platform_counts}))
display(pd.DataFrame({"sentiment_posts": sentiment_counts}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
city_counts.plot(kind="bar", ax=axes[0], title="Posts by city", color="#4c78a8")
platform_counts.plot(kind="bar", ax=axes[1], title="Posts by platform", color="#f58518")
sentiment_counts.plot(kind="bar", ax=axes[2], title="Posts by sentiment", color="#54a24b")
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("posts")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()

## 3. Fetch cleaned posts

In [ ]:
cities = ["sydney", "melbourne", "brisbane"]
frames = []

for city in cities:
    payload = api_get(mode="posts", city=city, size=200)
    rows = payload.get("rows", [])
    print(city, "rows:", len(rows), "reported total:", payload.get("total"))
    frames.append(pd.DataFrame(rows))

posts = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
require_frame(posts, ["id", "city", "platform", "date_local", "sentiment", "text", "tavg", "tmax", "prcp"], "Cleaned post sample")
posts.head()

In [ ]:
print("rows:", len(posts))
print("columns:", list(posts.columns))

needed = ["city", "platform", "date_local", "sentiment", "sentiment_label", "tavg", "tmax", "prcp", "text"]
posts[needed].head(10)

## 4. Simple analysis

In [ ]:
posts["sentiment"] = pd.to_numeric(posts["sentiment"], errors="coerce")
for col in ["tavg", "tmax", "prcp"]:
    if col in posts.columns:
        posts[col] = pd.to_numeric(posts[col], errors="coerce")

analysis = (
    posts.groupby(["city", "platform"], dropna=False)
    .agg(
        posts=("id", "count"),
        avg_sentiment=("sentiment", "mean"),
        avg_temp=("tavg", "mean"),
        avg_rain=("prcp", "mean"),
    )
    .reset_index()
    .sort_values(["city", "posts"], ascending=[True, False])
)

analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

posts.groupby("city")["sentiment"].mean().sort_values().plot(
    kind="barh", ax=axes[0], color="#4c78a8", title="Average sentiment by city"
)
posts.groupby("platform")["sentiment"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="#f58518", title="Average sentiment by platform"
)

for ax in axes:
    ax.set_xlabel("average sentiment")
    ax.set_ylabel("")

plt.tight_layout()

In [ ]:
if "tavg" in posts.columns:
    plot_df = posts.dropna(subset=["tavg", "sentiment"])
    ax = plot_df.plot.scatter(
        x="tavg", y="sentiment", c="#4c78a8", alpha=0.45,
        figsize=(7, 4), title="Temperature vs sentiment"
    )
    ax.set_xlabel("average daily temperature")
    ax.set_ylabel("sentiment")

## 5. Daily aggregation from the API

In [ ]:
daily_payload = api_get(mode="daily", city="sydney")
daily = pd.DataFrame(daily_payload.get("days", []))
daily.head()

In [ ]:
if not daily.empty:
    daily["date"] = pd.to_datetime(daily["date"])
    daily = daily.sort_values("date")
    ax = daily.tail(60).plot(
        x="date", y=["count", "sentiment_mean"], figsize=(10, 4),
        title="Sydney daily posts and sentiment"
    )
    ax.set_xlabel("date")
    plt.tight_layout()